In [ ]:
import json
import pathlib

import numpy as np


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# job name -> (file, factor from the file's `alpha_sq` to |alpha|^2).
# results_small.json predates the 2026-09 alpha_sq fix (docs/models_ats.md,
# section 2): its labels are twice the cat size, so 0.5 puts both files on one
# grid. That grid is nominal only -- see the truncation section below.
FILES = {
    "ref": (REPO / "results" / "sweep_parity_protected_ref.json", 1.0),
    "small": (REPO / "results" / "sweep_parity_protected_small.json", 1.0),
}


def _complex_hook(d):
    """{"real", "imag"} -> complex, as floquet_lindblad.io.from_jsonable does.

    That module imports jax, which this client does not have.
    """
    return complex(d["real"], d["imag"]) if d.keys() == {"real", "imag"} else d


def load_sweep(path, to_cat_size):
    with open(path) as f:
        pts = json.load(f, object_hook=_complex_hook)
    for r in pts:
        r["alpha_sq_file"] = r["alpha_sq"]
        r["alpha_sq"] = r["alpha_sq"] * to_cat_size
        # linspace leaves 0.30000000000000004 and the like; round so eps_p works as a key.
        r["eps_p"] = round(r["eps_p"], 12)
        if "params" in r:
            # The true cat size at leading order, whichever convention wrote the file.
            r["cat_size"] = r["params"]["epsilon_d"] / (2 * r["params"]["g"])
        for key in ("res_hist", "apply_hist", "mu_hist", "fock_weight_a", "fock_weight_b"):
            if key in r:
                r[key] = np.asarray(r[key])
    return pts


results = {name: load_sweep(path, f) for name, (path, f) in FILES.items()}

ALPHA_SQ_LIST = sorted({r["alpha_sq"] for res in results.values() for r in res})
EPS_P_LIST = sorted({r["eps_p"] for res in results.values() for r in res})

# The truncation offset of `small` from `ref`, read off the files rather than
# restated: it has to be the same at every alpha_sq for the comparison to mean much.
trunc = {name: {r["alpha_sq"]: (r["n_a"], r["n_b"]) for r in res}
         for name, res in results.items()}
shifts = {(trunc["small"][a][0] - t[0], trunc["small"][a][1] - t[1])
          for a, t in trunc["ref"].items() if a in trunc["small"]}
assert len(shifts) == 1, f"truncation offset varies with alpha_sq: {shifts}"
TRUNC_SHIFT = dict(zip(("n_a", "n_b"), shifts.pop()))

for name, (path, f) in FILES.items():
    res = results[name]
    cats = [r["cat_size"] for r in res if "cat_size" in r]
    print(f"{name:>5}: {len(res)} points from {path.name}  |  alpha_sq "
          f"{min(r['alpha_sq'] for r in res):g}..{max(r['alpha_sq'] for r in res):g}"
          f" (file labels x{1 / f:g})  |  true |alpha|^2 {min(cats):.4f}..{max(cats):.4f}")
print(f"eps_p: {', '.join(f'{e:.3f}' for e in EPS_P_LIST)}  |  small = ref "
      f"n_a {TRUNC_SHIFT['n_a']:+d}, n_b {TRUNC_SHIFT['n_b']:+d}")

In [ ]:
ok = {name: [r for r in res if "error" not in r] for name, res in results.items()}
bad = {name: [r for r in res if "error" in r] for name, res in results.items()}

for name in results:
    print(f"\n=== job '{name}' ===")
    print(f"{'alpha_sq':>8} {'|a|^2':>7} {'eps_p':>6} {'n_a x n_b':>10} {'kappa_b':>8} {'rate_bf':>14} "
          f"{'1-Re mu':>9} {'res Krylov':>10} {'res RQ':>9} {'blocks':>6} {'/rtol':>8} "
          f"{'apps':>5} {'cyc':>4} {'warm':>5} {'conv':>5} {'GPU s':>6}")
    for r in ok[name]:
        s = r["settings"]
        drop = r["n_blocks_final"] * r["gap"]
        print(f"{r['alpha_sq']:8.2f} {r['cat_size']:7.4f} {r['eps_p']:6.3f} {r['n_a']:4d} x{r['n_b']:4d} "
              f"{r['kappa_b']:8.5f} {r['rate_bf'].real:14.6e} {r['gap']:9.2e} "
              f"{r['res_rel']:10.2e} {r['res_rel_n']:9.2e} {r['n_blocks_final']:6d} "
              f"{drop / s['rtol_final']:8.1e} {r['n_apply']:5d} {r['n_cycles']:4d} "
              f"{str(r['warm_start']):>5} {('yes' if r['converged'] else 'NO'):>5} "
              f"{r['timings_s']['loop'] + r['timings_s']['polish']:6.1f}")
    for r in bad[name]:
        print(f"{r['alpha_sq']:8.2f} {r['eps_p']:6.3f}  FAILED: {r['error']}")

    stalled = [r for r in ok[name] if not r["converged"]]
    if stalled:
        print("not converged (residual above res_tol at the cycle budget): "
              + ", ".join(f"({r['alpha_sq']:.2f}, {r['eps_p']:.3f}) res {r['res_rel']:.1e} "
                          f"> {r['settings']['res_tol']:.0e}" for r in stalled))

    # The claim the per-band tolerances rest on: along each chain the gap grows,
    # so eps_p = EPS_P_LIST[0] is the hardest point. A shrinking gap means the
    # band was sized for the wrong end of that chain.
    for a in ALPHA_SQ_LIST:
        ch = [r for r in ok[name] if r["alpha_sq"] == a]
        if len(ch) > 1 and ch[-1]["gap"] <= ch[0]["gap"]:
            print(f"alpha_sq = {a:.2f}: gap SHRINKS along eps_p "
                  f"({ch[0]['gap']:.2e} -> {ch[-1]['gap']:.2e}) -- re-size its band")

### Truncation convergence: `ref` against `small`

At each grid point, the relative difference of the two rates. If the ladder's
truncation is converged, it sits at the level of the solver's own error (roughly
the RQ residual); if it is much larger, the truncation, not the solver, sets the
number -- and `ref` itself should not be trusted until a larger truncation agrees
with it.

**The two runs are not at the same cat size.** `small` ran before the 2026-09
`alpha_sq` fix, with `epsilon_d = 2 alpha_sq g2`: at the halved label, its true
`|alpha|^2 = epsilon_d / (2 g)` exceeds `ref`'s by the factor
`2 J_1(eps_p) / sin(eps_p)`, i.e. +0.04 % at `eps_p = 0.1` and +5.7 % at 1.1. The
rate falls roughly as `exp(-2 |alpha|^2)`, so even the smallest offset moves it by
more than `REL_TOL` at large `alpha_sq`, and the raw difference mostly measures the
offset. The verdict therefore compares `small` with `ref` *interpolated to
`small`'s true cat size* (cubic spline of `ln rate` along each `eps_p` chain; both
runs share `g` and `kappa_b`, so this reaches `small`'s physical point exactly).
Past `ref`'s last point it is an extrapolation, marked `extrap`. The clean test
is still to re-run `small` in the current convention.

In [ ]:
from scipy.interpolate import CubicSpline

REL_TOL = 1e-3  # agreement asked of the two truncations on the rate

by_point = {name: {(r["alpha_sq"], r["eps_p"]): r for r in ok[name]} for name in ok}
common_pts = sorted(set(by_point["ref"]) & set(by_point["small"]))

# ln(rate_ref) against the true cat size, one spline per eps_p chain.
ref_spline = {}
for e in EPS_P_LIST:
    ch = sorted((r for r in ok["ref"] if r["eps_p"] == e), key=lambda r: r["cat_size"])
    if len(ch) >= 4:
        spline = CubicSpline([r["cat_size"] for r in ch],
                             np.log([abs(r["rate_bf"].real) for r in ch]))
        ref_spline[e] = (spline, ch[0]["cat_size"], ch[-1]["cat_size"])

print(f"{'alpha_sq':>8} {'eps_p':>6} {'ref':>9} {'small':>9} {'d|a|^2':>7} {'rate ref':>13} "
      f"{'rate small':>13} {'raw diff':>9} {'matched':>9} {'res RQ ref':>10} {'verdict':>12}")
conv_rel, raw_rel, extrapolated = {}, {}, []
for pt in common_pts:
    r, s = by_point["ref"][pt], by_point["small"][pt]
    raw_rel[pt] = abs(s["rate_bf"].real - r["rate_bf"].real) / abs(r["rate_bf"].real)
    verdict = "n/a"
    if pt[1] in ref_spline:
        spline, lo, hi = ref_spline[pt[1]]
        ref_at_small = np.exp(spline(s["cat_size"]))
        conv_rel[pt] = abs(abs(s["rate_bf"].real) - ref_at_small) / ref_at_small
        verdict = "PASS" if conv_rel[pt] < REL_TOL else "FAIL"
        if not lo <= s["cat_size"] <= hi:
            extrapolated.append(pt)
            verdict += " extrap"
    print(f"{pt[0]:8.2f} {pt[1]:6.3f} {r['n_a']:4d}x{r['n_b']:<4d} {s['n_a']:4d}x{s['n_b']:<4d} "
          f"{s['cat_size'] - r['cat_size']:+7.4f} {r['rate_bf'].real:13.6e} "
          f"{s['rate_bf'].real:13.6e} {raw_rel[pt]:9.1e} {conv_rel.get(pt, np.nan):9.1e} "
          f"{r['res_rel_n']:10.1e} {verdict:>12}")
missing = sorted(set(by_point["ref"]) ^ set(by_point["small"]))
if missing:
    print("\nin one job only (failed in the other): "
          + ", ".join(f"({a:.2f}, {e:.3f})" for a, e in missing))
failed = [pt for pt, rel in conv_rel.items() if rel >= REL_TOL]
print(f"\nrate agreement at equal cat size against {REL_TOL:.0e}: "
      + ("all points PASS" if not failed else
         "truncation not converged at " + ", ".join(f"({a:.2f}, {e:.3f})" for a, e in failed)))
if extrapolated:
    print("extrapolated past ref's largest cat size (verdict less reliable): "
          + ", ".join(f"({a:.2f}, {e:.3f})" for a, e in extrapolated))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 3.8), constrained_layout=True)
cmap = plt.get_cmap("viridis")
for j, e in enumerate(EPS_P_LIST):
    pts = [pt for pt in common_pts if pt[1] == e]
    if not pts:
        continue
    c = cmap(j / max(1, len(EPS_P_LIST) - 1))
    x = [pt[0] for pt in pts]
    ax.plot(x, [conv_rel.get(pt, np.nan) for pt in pts], "o-", color=c,
            label=rf"$\varepsilon_p$ = {e:.2f}")
    ax.plot(x, [raw_rel[pt] for pt in pts], "o--", mfc="none", lw=0.8, color=c)
ax.plot([], [], "o--", mfc="none", lw=0.8, color="k", label="raw (unequal cat size)")
ax.axhline(REL_TOL, color="0.4", lw=0.8, ls=":", label="REL_TOL")
ax.set_yscale("log")
ax.set_xlabel(r"$|\alpha|^2$")
ax.set_ylabel(r"$|\Gamma_{small} - \Gamma_{ref}| / \Gamma_{ref}$")
ax.set_title(f"truncation shift n_a {TRUNC_SHIFT['n_a']:+d}, n_b {TRUNC_SHIFT['n_b']:+d}"
             " (filled: at equal cat size)", fontsize=9)
ax.legend(fontsize=7, frameon=False)
ax.grid(alpha=0.3)
plt.show()

### Bit-flip rate on the grid

Left: against the true cat size `|alpha|^2 = epsilon_d / (2 g)`, one colour per
`eps_p` (filled: `ref`, open: `small`, which sits slightly to the right at large
`eps_p`, see above), with the perturbative estimate of the `alpha_sq` sweep.
Right: against pump strength, one colour per `alpha_sq` (`ref` only).

In [ ]:
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

# Left: rate vs the true |alpha|^2 = epsilon_d / (2 g) (docs/models_ats.md, section 2):
# the file labels do not agree between the two runs.
cmap_e = plt.get_cmap("viridis")
for j, e in enumerate(EPS_P_LIST):
    c = cmap_e(j / max(1, len(EPS_P_LIST) - 1))
    for name, style in (("ref", dict(color=c)), ("small", dict(facecolors="none", edgecolors=c))):
        pts = [r for r in ok[name] if r["eps_p"] == e]
        if not pts:
            continue
        n = np.array([r["cat_size"] for r in pts])
        axes[0].scatter(n, [r["rate_bf"].real for r in pts], zorder=3,
                        label=rf"$\varepsilon_p$ = {e:.2f}" if name == "ref" else None,
                        **style)
        if name == "ref":
            # Perturbative estimate, verbatim from the alpha_sq sweep, including
            # its x3000 eyeball normalisation (set at eps_p = 0.1): a shape
            # comparison, not a calibrated prefactor. kappa_1 / kappa_2 come
            # back in `params`, derived on the worker from floquet_lindblad.models.ats.
            k1 = np.array([r["params"]["kappa_1"] for r in pts])
            k2 = np.array([r["params"]["kappa_2"] for r in pts])
            pert_th = k1 * n * np.exp(-4 * n) + k1**2 / (2 * k2) * np.exp(-2 * n)
            axes[0].plot(n, pert_th * 3000, ls=":", color=c)
axes[0].plot([], [], ls=":", c="k", label=r"perturbative $\times$ 3000")
axes[0].scatter([], [], facecolors="none", edgecolors="k", label="small truncation")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"$|\alpha|^2$")
axes[0].set_ylabel("bit-flip rate (GHz)")
axes[0].legend(fontsize=7, frameon=False)

# Right: rate vs eps_p, one line per alpha_sq chain.
cmap_a = mpl.colormaps["plasma"]
norm_a = mpl.colors.Normalize(vmin=min(ALPHA_SQ_LIST), vmax=max(ALPHA_SQ_LIST))
for a in ALPHA_SQ_LIST:
    pts = [r for r in ok["ref"] if r["alpha_sq"] == a]
    if pts:
        axes[1].plot([r["eps_p"] for r in pts], [r["rate_bf"].real for r in pts], "o-",
                     color=cmap_a(norm_a(a)))
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=cmap_a), ax=axes[1],
             label=r"$\alpha_{sq}$")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"$\varepsilon_p$")
axes[1].set_ylabel("bit-flip rate (GHz)")

for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

# Left: rate vs the true |alpha|^2 = epsilon_d / (2 g) (docs/models_ats.md, section 2):
# the file labels do not agree between the two runs.
cmap_e = plt.get_cmap("viridis")
for j, e in enumerate(EPS_P_LIST):
    c = cmap_e(j / max(1, len(EPS_P_LIST) - 1))
    for name, style in (("ref", dict(color=c)), ("small", dict(facecolors="none", edgecolors=c))):
        pts = [r for r in ok[name] if r["eps_p"] == e]
        if not pts:
            continue
        n = np.array([r["cat_size"] for r in pts])
        axes[0].scatter(n, [np.abs(r["mu"].imag) for r in pts], zorder=3,
                        label=rf"$\varepsilon_p$ = {e:.2f}" if name == "ref" else None,
                        **style)
axes[0].plot([], [], ls=":", c="k", label=r"perturbative $\times$ 3000")
axes[0].scatter([], [], facecolors="none", edgecolors="k", label="small truncation")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"$|\alpha|^2$")
axes[0].set_ylabel("bit-flip rate (GHz)")
axes[0].legend(fontsize=7, frameon=False)

# Right: rate vs eps_p, one line per alpha_sq chain.
cmap_a = mpl.colormaps["plasma"]
norm_a = mpl.colors.Normalize(vmin=min(ALPHA_SQ_LIST), vmax=max(ALPHA_SQ_LIST))
for a in ALPHA_SQ_LIST:
    pts = [r for r in ok["ref"] if r["alpha_sq"] == a]
    if pts:
        axes[1].plot([r["eps_p"] for r in pts], [np.abs(r["mu"].imag) for r in pts], "o-",
                     color=cmap_a(norm_a(a)))
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=cmap_a), ax=axes[1],
             label=r"$\alpha_{sq}$")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"$\varepsilon_p$")
axes[1].set_ylabel("bit-flip rate (GHz)")

for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

### Adiabatic ratio

With `KEEP_ADIABATIC_RATIO`, `kappa_b / g` should be constant along each
`eps_p` chain by construction (that is what the `kappa_b` rescaling is for),
and identical across chains. With a fixed `kappa_b` it falls as
`1 / sin(eps_p)`, and the buffer occupation `(epsilon_d / kappa_b)**2` grows
as `sin(eps_p)**2`: check `tail_b` at the largest `eps_p`.

In [ ]:
print(f"{'alpha_sq':>8}  kappa_b / g along eps_p")
for a in ALPHA_SQ_LIST:
    pts = [r for r in ok["ref"] if r["alpha_sq"] == a]
    # g comes back in `params`; kappa_b and g are both in the model's units.
    ratio = [r["kappa_b"] / r["params"]["g"] for r in pts]
    print(f"{a:8.2f}  " + "  ".join(f"{x:.4f}" for x in ratio))

### What the warm start bought, and convergence per chain

Left: the target residual per cycle, one colour per `alpha_sq`, dashed for the
cold first point of each chain -- the warm points should start a decade or more
lower and converge in fewer cycles. Right: period integrations per point. Run
again with `WARM_START = False` for the control; compare `n_apply` and GPU
seconds, not wall clock.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), constrained_layout=True)

for a in ALPHA_SQ_LIST:
    c = cmap_a(norm_a(a))
    for r in (r for r in ok["ref"] if r["alpha_sq"] == a):
        axes[0].plot(r["apply_hist"], r["res_hist"], marker=".", ms=3, lw=1.0, color=c,
                     ls="-" if r["warm_start"] else "--")
        axes[0].plot(r["apply_hist"][-1], r["settings"]["res_tol"], "_", ms=8, color=c)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=cmap_a), ax=axes[0],
             label=r"$\alpha_{sq}$")
axes[0].set_yscale("log")
axes[0].set_xlabel("period integrations")
axes[0].set_ylabel(r"residual $\|U y - \theta y\| / |\theta|$")
axes[0].set_title("job 'ref' (dashed: cold start, tick: res_tol)", fontsize=9)

n_eps = len(EPS_P_LIST)
w = 0.8 / n_eps
x = np.arange(len(ALPHA_SQ_LIST))
for j, e in enumerate(EPS_P_LIST):
    apps = [next((r["n_apply"] for r in ok["ref"]
                  if r["alpha_sq"] == a and r["eps_p"] == e), np.nan) for a in ALPHA_SQ_LIST]
    axes[1].bar(x + (j - (n_eps - 1) / 2) * w, apps, width=w,
                color=cmap_e(j / max(1, n_eps - 1)), label=rf"$\varepsilon_p$ = {e:.2f}")
axes[1].set_xticks(x, [f"{a:g}" for a in ALPHA_SQ_LIST])
axes[1].set_xlabel(r"$\alpha_{sq}$")
axes[1].set_ylabel("period integrations")
axes[1].set_title("cost per point, job 'ref'", fontsize=9)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

for name in ok:
    apps = np.array([r["n_apply"] for r in ok[name]])
    warm = np.array([r["warm_start"] for r in ok[name]])
    if warm.any() and (~warm).any():
        print(f"[{name}] cold: {apps[~warm].mean():.0f} period integrations on average, "
              f"warm: {apps[warm].mean():.0f}  "
              f"(saving on a warm point: {1 - apps[warm].mean() / apps[~warm].mean():+.0%})")
    elif len(apps):
        print(f"[{name}] all points {'warm' if warm.all() else 'cold'}: "
              f"{apps.mean():.0f} period integrations on average")

### Fock-tail check

The criterion of the single-point notebooks, for both jobs: the weight in the
top `k_top` levels of each mode has to sit far below the residual being quoted.
It is necessary, not sufficient -- the `ref` / `small` comparison above is the
sharper test.

In [ ]:
TOL = 1e-11  # same absolute criterion as the single-point notebooks

for name in ok:
    print(f"\n=== job '{name}' ===")
    print(f"{'alpha_sq':>8} {'eps_p':>6} {'n_a':>4} {'n_b':>4} {'tail_a':>10} {'tail_b':>10} "
          f"{'res RQ':>10} {'verdict':>8}")
    worst = []
    for r in ok[name]:
        passed = max(r["tail_a"], r["tail_b"]) < TOL
        if not passed:
            worst.append(r)
        print(f"{r['alpha_sq']:8.2f} {r['eps_p']:6.3f} {r['n_a']:4d} {r['n_b']:4d} "
              f"{r['tail_a']:10.2e} {r['tail_b']:10.2e} {r['res_rel_n']:10.2e} "
              f"{('PASS' if passed else 'FAIL'):>8}")
    if ok[name]:
        print(f"top-{ok[name][0]['k_top']} weight against tol = {TOL:.0e}: "
              + ("all points PASS" if not worst else
                 "FAIL at " + ", ".join(f"({r['alpha_sq']:.2f}, {r['eps_p']:.3f})"
                                        for r in worst)))